# 02 — Data Cleaning & Panel Construction

This notebook takes the raw data collected in `01_data_collection.ipynb` and:
1. Cleans and standardizes each dataset
2. Merges everything into a state × year panel (2010–2022)
3. Constructs treatment variables for DiD analysis
4. Validates the final panel

**Input:** Files in `data/raw/`  
**Output:** `data/processed/analysis_panel.csv`

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

RAW_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Study parameters
YEAR_START = 2010
YEAR_END = 2022
YEARS = list(range(YEAR_START, YEAR_END + 1))

print(f"Panel period: {YEAR_START}–{YEAR_END} ({len(YEARS)} years)")

Panel period: 2010–2022 (13 years)


---
## 1. Load & Clean Medicaid Expansion Status

In [2]:
df_expansion = pd.read_csv(f'{RAW_DIR}/kff_expansion_status.csv')

print(f"Shape: {df_expansion.shape}")
print(f"\nExpansion breakdown:")
print(df_expansion['cohort'].value_counts())

# Ensure state_fips is zero-padded string
df_expansion['state_fips'] = df_expansion['state_fips'].astype(str).str.zfill(2)

df_expansion.head()

Shape: (51, 6)

Expansion breakdown:
cohort
Early (2014)    27
Never           10
Late (2015)      3
Late (2020)      3
Late (2016)      2
Late (2019)      2
Late (2021)      2
Late (2023)      2
Name: count, dtype: int64


,state,state_fips,expansion_year,expansion_date,ever_expanded,cohort
0,Alabama,01,0,NaN,0,Never
1,Alaska,02,2015,2015-09-01,1,Late (2015)
2,Arizona,04,2014,2014-01-01,1,Early (2014)
3,Arkansas,05,2014,2014-01-01,1,Early (2014)
4,California,06,2014,2014-01-01,1,Early (2014)


---
## 2. Create the Base Panel Skeleton

Create a balanced panel with every state × year combination, then merge treatment variables.

In [3]:
# Create state × year skeleton
states = df_expansion[['state', 'state_fips']].copy()
years_df = pd.DataFrame({'year': YEARS})

# Cross join
panel = states.merge(years_df, how='cross')
print(f"Panel skeleton: {panel.shape[0]} rows ({len(states)} states × {len(YEARS)} years)")

# Merge expansion status
panel = panel.merge(
    df_expansion[['state_fips', 'expansion_year', 'ever_expanded', 'cohort']],
    on='state_fips',
    how='left'
)

# Construct treatment timing variables
panel['post_expansion'] = np.where(
    (panel['ever_expanded'] == 1) & (panel['year'] >= panel['expansion_year']),
    1, 0
)

# Years relative to expansion (event time)
# For never-treated states, this stays NaN
panel['event_time'] = np.where(
    panel['ever_expanded'] == 1,
    panel['year'] - panel['expansion_year'],
    np.nan
)

# Cohort group for Callaway-Sant'Anna (0 = never treated)
panel['cohort_group'] = np.where(
    panel['ever_expanded'] == 1,
    panel['expansion_year'],
    0
)

# DiD interaction term (for simple 2x2 DiD)
panel['treat_x_post'] = panel['ever_expanded'] * panel['post_expansion']

print(f"\nTreatment variable check:")
print(f"  post_expansion = 1: {panel['post_expansion'].sum()} obs")
print(f"  post_expansion = 0: {(panel['post_expansion'] == 0).sum()} obs")
print(f"  Ever expanded: {panel['ever_expanded'].sum()} obs")
print(f"  Never expanded: {(panel['ever_expanded'] == 0).sum()} obs")

panel.head(10)

Panel skeleton: 663 rows (51 states × 13 years)

Treatment variable check:
  post_expansion = 1: 302 obs
  post_expansion = 0: 361 obs
  Ever expanded: 533 obs
  Never expanded: 130 obs


,state,state_fips,year,expansion_year,ever_expanded,cohort,post_expansion,event_time,cohort_group,treat_x_post
0,Alabama,01,2010,0,0,Never,0,NaN,0,0
1,Alabama,01,2011,0,0,Never,0,NaN,0,0
2,Alabama,01,2012,0,0,Never,0,NaN,0,0
3,Alabama,01,2013,0,0,Never,0,NaN,0,0
4,Alabama,01,2014,0,0,Never,0,NaN,0,0
5,Alabama,01,2015,0,0,Never,0,NaN,0,0
6,Alabama,01,2016,0,0,Never,0,NaN,0,0
7,Alabama,01,2017,0,0,Never,0,NaN,0,0
8,Alabama,01,2018,0,0,Never,0,NaN,0,0
9,Alabama,01,2019,0,0,Never,0,NaN,0,0


In [4]:
# Validate: check a known state
print("Louisiana (expanded July 2016):")
la = panel[panel['state'] == 'Louisiana'][['state', 'year', 'expansion_year', 'post_expansion', 'event_time']]
display(la)

print("\nTexas (never expanded):")
tx = panel[panel['state'] == 'Texas'][['state', 'year', 'expansion_year', 'post_expansion', 'event_time']]
display(tx)

Louisiana (expanded July 2016):


,state,year,expansion_year,post_expansion,event_time
234,Louisiana,2010,2016,0,-6.0
235,Louisiana,2011,2016,0,-5.0
236,Louisiana,2012,2016,0,-4.0
237,Louisiana,2013,2016,0,-3.0
238,Louisiana,2014,2016,0,-2.0
239,Louisiana,2015,2016,0,-1.0
240,Louisiana,2016,2016,1,0.0
241,Louisiana,2017,2016,1,1.0
242,Louisiana,2018,2016,1,2.0
243,Louisiana,2019,2016,1,3.0



Texas (never expanded):


,state,year,expansion_year,post_expansion,event_time
559,Texas,2010,0,0,NaN
560,Texas,2011,0,0,NaN
561,Texas,2012,0,0,NaN
562,Texas,2013,0,0,NaN
563,Texas,2014,0,0,NaN
564,Texas,2015,0,0,NaN
565,Texas,2016,0,0,NaN
566,Texas,2017,0,0,NaN
567,Texas,2018,0,0,NaN
568,Texas,2019,0,0,NaN


---
## 3. Clean & Merge ACS Controls

In [5]:
acs_path = f'{RAW_DIR}/acs_state_controls.csv'

if os.path.exists(acs_path):
    df_acs = pd.read_csv(acs_path)
    df_acs['state_fips'] = df_acs['state_fips'].astype(str).str.zfill(2)
    
    print(f"ACS data: {df_acs.shape}")
    print(f"Years: {df_acs['year'].min()}–{df_acs['year'].max()}")
    print(f"\nMissing values:")
    print(df_acs.isnull().sum())
    
    # Note: ACS 2020 1-year was not released due to COVID.
    # Check if 2020 is missing and interpolate if needed.
    years_present = sorted(df_acs['year'].unique())
    missing_years = [y for y in YEARS if y not in years_present]
    if missing_years:
        print(f"\n⚠️  Missing ACS years: {missing_years}")
        print("Will interpolate these after merging.")
    
    # Select columns for merge
    acs_cols = ['state_fips', 'year', 'total_population', 'median_household_income',
                'poverty_rate', 'pct_white', 'pct_black', 'pct_hispanic']
    acs_merge = df_acs[[c for c in acs_cols if c in df_acs.columns]].copy()
    
    # Merge to panel
    panel = panel.merge(acs_merge, on=['state_fips', 'year'], how='left')
    
    # Interpolate missing years (e.g., 2020) within each state
    if missing_years:
        numeric_acs = [c for c in acs_merge.columns if c not in ['state_fips', 'year']]
        panel = panel.sort_values(['state_fips', 'year'])
        panel[numeric_acs] = panel.groupby('state_fips')[numeric_acs].transform(
            lambda x: x.interpolate(method='linear')
        )
        print(f"Interpolated {missing_years} for ACS variables.")
    
    print(f"\nPanel after ACS merge: {panel.shape}")
else:
    print(f"⚠️  ACS file not found at {acs_path}")
    print("Run 01_data_collection.ipynb first.")

⚠️  ACS file not found at ../data/raw/acs_state_controls.csv
Run 01_data_collection.ipynb first.


---
## 4. Clean & Merge Mortality Data

In [6]:
def clean_cdc_wonder(filepath, rate_prefix):
    """
    Parse and clean a CDC WONDER mortality export file.
    
    Args:
        filepath: Path to the .txt export
        rate_prefix: Prefix for column names (e.g., 'allcause', 'diabetes')
    
    Returns:
        DataFrame with state_fips, year, deaths, crude_rate, age_adj_rate
    """
    if not os.path.exists(filepath):
        print(f"  ⚠️  Not found: {filepath}")
        return None
    
    # Read tab-delimited, skip metadata rows at bottom
    rows = []
    with open(filepath, 'r') as f:
        header = f.readline().strip().split('\t')
        header = [h.strip('"') for h in header]
        for line in f:
            stripped = line.strip()
            if stripped.startswith('---') or stripped.startswith('"---') or not stripped:
                break
            vals = [v.strip('"') for v in stripped.split('\t')]
            rows.append(vals)
    
    df = pd.DataFrame(rows, columns=header)
    
    # Standardize column names (CDC WONDER uses various names)
    col_map = {}
    for col in df.columns:
        col_lower = col.lower()
        if 'state code' in col_lower:
            col_map[col] = 'state_fips'
        elif col_lower == 'state':
            col_map[col] = 'state_name'
        elif col_lower in ['year', 'year code']:
            col_map[col] = 'year'
        elif col_lower == 'deaths':
            col_map[col] = f'{rate_prefix}_deaths'
        elif col_lower == 'population':
            col_map[col] = f'{rate_prefix}_population'
        elif 'crude rate' in col_lower:
            col_map[col] = f'{rate_prefix}_crude_rate'
        elif 'age adjusted' in col_lower:
            col_map[col] = f'{rate_prefix}_age_adj_rate'
    
    df = df.rename(columns=col_map)
    
    # Convert types
    if 'state_fips' in df.columns:
        df['state_fips'] = df['state_fips'].astype(str).str.zfill(2)
    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
    
    # Convert numeric columns, handling suppressed/unreliable values
    suppressed_values = ['Suppressed', 'Unreliable', 'Not Applicable', '']
    for col in df.columns:
        if rate_prefix in col and col not in ['state_fips', 'year', 'state_name']:
            df[col] = df[col].replace(suppressed_values, np.nan)
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Keep only needed columns
    keep = ['state_fips', 'year'] + [c for c in df.columns if rate_prefix in c]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].copy()
    
    # Drop rows with missing key identifiers
    df = df.dropna(subset=['state_fips', 'year'])
    df['year'] = df['year'].astype(int)
    
    print(f"  ✅ {rate_prefix}: {len(df)} rows, cols = {[c for c in df.columns if rate_prefix in c]}")
    return df

In [7]:
# Clean and merge each mortality dataset
print("Cleaning mortality data...\n")

mortality_configs = [
    ('cdc_wonder_allcause_mortality.txt', 'allcause'),
    ('cdc_wonder_diabetes_mortality.txt', 'diabetes'),
    ('cdc_wonder_maternal_mortality.txt', 'maternal'),
]

for filename, prefix in mortality_configs:
    filepath = f'{RAW_DIR}/{filename}'
    df_mort = clean_cdc_wonder(filepath, prefix)
    
    if df_mort is not None:
        panel = panel.merge(df_mort, on=['state_fips', 'year'], how='left')

print(f"\nPanel after mortality merge: {panel.shape}")

Cleaning mortality data...

  ⚠️  Not found: ../data/raw/cdc_wonder_allcause_mortality.txt
  ⚠️  Not found: ../data/raw/cdc_wonder_diabetes_mortality.txt
  ⚠️  Not found: ../data/raw/cdc_wonder_maternal_mortality.txt

Panel after mortality merge: (663, 10)


---
## 5. Clean & Merge Diabetes Surveillance Data

In [8]:
def clean_diabetes_atlas(filepath, measure_name):
    """
    Clean CDC Diabetes Atlas download.
    These files vary in format — this function handles common patterns.
    """
    if not os.path.exists(filepath):
        print(f"  ⚠️  Not found: {filepath}")
        return None
    
    df = pd.read_csv(filepath)
    print(f"  Raw columns: {df.columns.tolist()}")
    
    # Try to identify state, year, and value columns
    col_map = {}
    for col in df.columns:
        col_lower = col.lower().strip()
        if 'state' in col_lower or 'location' in col_lower:
            col_map[col] = 'state'
        elif 'year' in col_lower:
            col_map[col] = 'year'
        elif 'fips' in col_lower:
            col_map[col] = 'state_fips'
        elif any(kw in col_lower for kw in ['percentage', 'prevalence', 'rate', 'number', 'incidence']):
            col_map[col] = f'diabetes_{measure_name}'
    
    df = df.rename(columns=col_map)
    
    if 'state_fips' in df.columns:
        df['state_fips'] = df['state_fips'].astype(str).str.zfill(2)
    
    print(f"  ✅ {measure_name}: {len(df)} rows")
    return df

# Try to load and merge diabetes data
print("Cleaning diabetes data...\n")

diabetes_configs = [
    ('cdc_diabetes_prevalence.csv', 'prevalence'),
    ('cdc_diabetes_incidence.csv', 'incidence'),
]

for filename, measure in diabetes_configs:
    filepath = f'{RAW_DIR}/{filename}'
    df_diab = clean_diabetes_atlas(filepath, measure)
    
    if df_diab is not None:
        if 'state_fips' in df_diab.columns:
            merge_cols = ['state_fips', 'year']
        elif 'state' in df_diab.columns:
            merge_cols = ['state', 'year']
        else:
            print(f"  ⚠️  Cannot identify merge key for {filename}")
            continue
        
        value_cols = [c for c in df_diab.columns if 'diabetes' in c]
        df_diab_merge = df_diab[merge_cols + value_cols].copy()
        panel = panel.merge(df_diab_merge, on=merge_cols, how='left')

print(f"\nPanel after diabetes merge: {panel.shape}")

Cleaning diabetes data...

  ⚠️  Not found: ../data/raw/cdc_diabetes_prevalence.csv
  ⚠️  Not found: ../data/raw/cdc_diabetes_incidence.csv

Panel after diabetes merge: (663, 10)


---
## 6. Clean & Merge Natality Data

In [9]:
natality_path = f'{RAW_DIR}/cdc_wonder_natality.txt'

if os.path.exists(natality_path):
    rows = []
    with open(natality_path, 'r') as f:
        header = [h.strip('"') for h in f.readline().strip().split('\t')]
        for line in f:
            stripped = line.strip()
            if stripped.startswith('---') or stripped.startswith('"---') or not stripped:
                break
            vals = [v.strip('"') for v in stripped.split('\t')]
            rows.append(vals)
    
    df_natality = pd.DataFrame(rows, columns=header)
    print(f"Natality raw: {df_natality.shape}")
    print(f"Columns: {df_natality.columns.tolist()}")
    
    # Rename key columns (adjust after seeing actual download)
    col_map = {}
    for col in df_natality.columns:
        col_lower = col.lower()
        if 'state code' in col_lower:
            col_map[col] = 'state_fips'
        elif col_lower == 'year' or 'year code' in col_lower:
            col_map[col] = 'year'
        elif 'births' in col_lower and 'birth weight' not in col_lower:
            col_map[col] = 'total_births'
        elif 'lbw' in col_lower or 'low birth weight' in col_lower:
            col_map[col] = 'low_birth_weight_pct'
        elif 'average birth weight' in col_lower:
            col_map[col] = 'avg_birth_weight'
    
    df_natality = df_natality.rename(columns=col_map)
    
    if 'state_fips' in df_natality.columns:
        df_natality['state_fips'] = df_natality['state_fips'].astype(str).str.zfill(2)
        df_natality['year'] = pd.to_numeric(df_natality['year'], errors='coerce')
        
        for col in ['total_births', 'low_birth_weight_pct', 'avg_birth_weight']:
            if col in df_natality.columns:
                df_natality[col] = pd.to_numeric(
                    df_natality[col].replace(['Suppressed', 'Unreliable', ''], np.nan),
                    errors='coerce'
                )
        
        natality_cols = ['state_fips', 'year'] + [
            c for c in ['total_births', 'low_birth_weight_pct', 'avg_birth_weight'] 
            if c in df_natality.columns
        ]
        df_natality_merge = df_natality[natality_cols].dropna(subset=['state_fips', 'year'])
        df_natality_merge['year'] = df_natality_merge['year'].astype(int)
        
        panel = panel.merge(df_natality_merge, on=['state_fips', 'year'], how='left')
        print(f"✅ Merged natality data. Panel: {panel.shape}")
    else:
        print("⚠️  Could not identify state_fips column. Check column names and adjust.")
else:
    print(f"⚠️  Natality file not found at {natality_path}")

⚠️  Natality file not found at ../data/raw/cdc_wonder_natality.txt


---
## 7. Clean & Merge BRFSS Health Access Data

In [10]:
brfss_path = f'{RAW_DIR}/brfss_health_access.csv'

if os.path.exists(brfss_path):
    df_brfss = pd.read_csv(brfss_path)
    print(f"BRFSS raw: {df_brfss.shape}")
    print(f"Columns: {df_brfss.columns.tolist()[:15]}")
    
    # BRFSS format varies depending on download method
    # Example for CDC SODAPI format:
    if 'locationabbr' in df_brfss.columns:
        try:
            import us
            state_abbrevs = [s.abbr for s in us.states.STATES_AND_DC]
            df_brfss = df_brfss[df_brfss['locationabbr'].isin(state_abbrevs)].copy()
            abbr_to_fips = {s.abbr: s.fips for s in us.states.STATES_AND_DC}
            df_brfss['state_fips'] = df_brfss['locationabbr'].map(abbr_to_fips)
        except ImportError:
            print("Install 'us' package: pip install us")
    
    print("\n⚠️  BRFSS data format varies. Review columns above and adjust as needed.")
    display(df_brfss.head())
else:
    print(f"⚠️  BRFSS file not found. This is optional — proceeding without it.")

⚠️  BRFSS file not found. This is optional — proceeding without it.


---
## 8. Final Panel Validation

In [11]:
print("=" * 60)
print("FINAL PANEL SUMMARY")
print("=" * 60)

print(f"\nShape: {panel.shape}")
print(f"States: {panel['state'].nunique()}")
print(f"Years: {panel['year'].min()}–{panel['year'].max()}")
print(f"Obs per state: {panel.groupby('state').size().unique()}")

print(f"\nColumns ({len(panel.columns)}):")
for col in panel.columns:
    non_null = panel[col].notna().sum()
    pct = non_null / len(panel) * 100
    print(f"  {col:40s} {non_null:>5d} non-null ({pct:5.1f}%)")

FINAL PANEL SUMMARY

Shape: (663, 10)
States: 51
Years: 2010–2022
Obs per state: [13]

Columns (10):
  state                                      663 non-null (100.0%)
  state_fips                                 663 non-null (100.0%)
  year                                       663 non-null (100.0%)
  expansion_year                             663 non-null (100.0%)
  ever_expanded                              663 non-null (100.0%)
  cohort                                     663 non-null (100.0%)
  post_expansion                             663 non-null (100.0%)
  event_time                                 533 non-null ( 80.4%)
  cohort_group                               663 non-null (100.0%)
  treat_x_post                               663 non-null (100.0%)


In [12]:
# Check balance
expected_obs = panel['state'].nunique() * len(YEARS)
actual_obs = len(panel)

if actual_obs == expected_obs:
    print(f"✅ Panel is balanced: {actual_obs} observations (expected {expected_obs})")
else:
    print(f"⚠️  Panel is UNBALANCED: {actual_obs} observations (expected {expected_obs})")
    state_year_counts = panel.groupby('state').size()
    problem_states = state_year_counts[state_year_counts != len(YEARS)]
    if len(problem_states) > 0:
        print(f"States with missing years: {problem_states.to_dict()}")

✅ Panel is balanced: 663 observations (expected 663)


In [13]:
# Treatment balance check
print("Treatment status by year:\n")
treat_by_year = panel.groupby('year').agg(
    n_treated=('post_expansion', 'sum'),
    n_untreated=('post_expansion', lambda x: (x == 0).sum()),
    n_never_expanded=('ever_expanded', lambda x: (x == 0).sum())
).reset_index()

print(treat_by_year.to_string(index=False))

Treatment status by year:

 year  n_treated  n_untreated  n_never_expanded
 2010          0           51                10
 2011          0           51                10
 2012          0           51                10
 2013          0           51                10
 2014         27           24                10
 2015         30           21                10
 2016         32           19                10
 2017         32           19                10
 2018         32           19                10
 2019         34           17                10
 2020         37           14                10
 2021         39           12                10
 2022         39           12                10


In [14]:
# Descriptive statistics by expansion status
print("\nDescriptive Statistics: Expansion vs Non-Expansion States (Pre-period, 2010-2013)\n")

pre_period = panel[panel['year'].between(2010, 2013)].copy()

# Identify numeric columns (excluding identifiers and treatment vars)
exclude_cols = ['state', 'state_fips', 'year', 'expansion_year', 'ever_expanded', 
                'cohort', 'post_expansion', 'event_time', 'cohort_group', 'treat_x_post']
numeric_cols = [c for c in pre_period.select_dtypes(include=[np.number]).columns 
                if c not in exclude_cols]

if numeric_cols:
    comparison = pre_period.groupby('ever_expanded')[numeric_cols].mean().T
    comparison.columns = ['Non-Expansion', 'Expansion']
    comparison['Difference'] = comparison['Expansion'] - comparison['Non-Expansion']
    display(comparison.round(2))
else:
    print("No outcome data available yet. Download CDC data and re-run.")


Descriptive Statistics: Expansion vs Non-Expansion States (Pre-period, 2010-2013)

No outcome data available yet. Download CDC data and re-run.


---
## 9. Save Final Panel

In [15]:
# Sort and save
panel = panel.sort_values(['state', 'year']).reset_index(drop=True)

output_path = f'{PROCESSED_DIR}/analysis_panel.csv'
panel.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / 1024
print(f"✅ Saved final panel to {output_path}")
print(f"   Size: {file_size:.1f} KB")
print(f"   Shape: {panel.shape}")
print(f"   Columns: {list(panel.columns)}")

✅ Saved final panel to ../data/processed/analysis_panel.csv
   Size: 31.6 KB
   Shape: (663, 10)
   Columns: ['state', 'state_fips', 'year', 'expansion_year', 'ever_expanded', 'cohort', 'post_expansion', 'event_time', 'cohort_group', 'treat_x_post']


In [16]:
# Final peek
print("Sample — Ohio (expanded 2014):\n")
display(panel[panel['state'] == 'Ohio'].head())

print("\nSample — Missouri (expanded 2021):\n")
display(panel[panel['state'] == 'Missouri'].head())

print("\nSample — Florida (never expanded):\n")
display(panel[panel['state'] == 'Florida'].head())

Sample — Ohio (expanded 2014):



,state,state_fips,year,expansion_year,ever_expanded,cohort,post_expansion,event_time,cohort_group,treat_x_post
455,Ohio,39,2010,2014,1,Early (2014),0,-4.0,2014,0
456,Ohio,39,2011,2014,1,Early (2014),0,-3.0,2014,0
457,Ohio,39,2012,2014,1,Early (2014),0,-2.0,2014,0
458,Ohio,39,2013,2014,1,Early (2014),0,-1.0,2014,0
459,Ohio,39,2014,2014,1,Early (2014),1,0.0,2014,1



Sample — Missouri (expanded 2021):



,state,state_fips,year,expansion_year,ever_expanded,cohort,post_expansion,event_time,cohort_group,treat_x_post
325,Missouri,29,2010,2021,1,Late (2021),0,-11.0,2021,0
326,Missouri,29,2011,2021,1,Late (2021),0,-10.0,2021,0
327,Missouri,29,2012,2021,1,Late (2021),0,-9.0,2021,0
328,Missouri,29,2013,2021,1,Late (2021),0,-8.0,2021,0
329,Missouri,29,2014,2021,1,Late (2021),0,-7.0,2021,0



Sample — Florida (never expanded):



,state,state_fips,year,expansion_year,ever_expanded,cohort,post_expansion,event_time,cohort_group,treat_x_post
117,Florida,12,2010,0,0,Never,0,NaN,0,0
118,Florida,12,2011,0,0,Never,0,NaN,0,0
119,Florida,12,2012,0,0,Never,0,NaN,0,0
120,Florida,12,2013,0,0,Never,0,NaN,0,0
121,Florida,12,2014,0,0,Never,0,NaN,0,0


---
## Next Steps

The panel is ready. Proceed to:
- **`03_eda.ipynb`** — Exploratory data analysis, descriptive statistics, pre-trends visualization
- **`04_did_analysis.ipynb`** — Difference-in-differences estimation